<a href="https://colab.research.google.com/github/sofialindner/ntl-urban-expansion-mapping/blob/develop/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import rasterio
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob

# Carrega lista de imagens
image_paths = glob.glob('viirs_images/*.tif')

# Cria série temporal para cada pixel, indexado por (linha, coluna)
pixel_series = {}

for path in image_paths:

  # Abre imagem .tif, preservando metadados
  with rasterio.open(path) as src:
    img = src.read(1, masked=True)

  # Filtra pixels válidos que de fato pertencem ao litoral
  valid_mask = ~img.mask
  rows, cols = np.where(valid_mask)

  # Percorre os pixels válidos e armazena a intensidade
  for row, col in zip(rows, cols):
      value = img[row, col]

      if np.isnan(value):
        continue

      key = (row, col)
      if key not in pixel_series:
          pixel_series[key] = []

      pixel_series[key].append(float(value))

In [25]:
from sklearn.linear_model import LinearRegression

def pixel_linear_trend(serie):
    y = np.array(serie)

    # Filtra valores NaN
    valid = ~np.isnan(y)

    y = y[valid]

    # Não tenta calcular regressão para séries pequenas
    if len(y) < 2:
        return 0.0

    X = np.arange(len(y)).reshape(-1,1)

    model = LinearRegression()
    model.fit(X, y)

    return model.coef_[0]

In [26]:
from dataclasses import dataclass

# Calcula features para cada pixel
pixel_features = {}

for key, serie in pixel_series.items():
  pixel_features[key] = {
      "mean": np.mean(serie),
      "var": np.var(serie),
      "max": np.max(serie),
      "min": np.min(serie),
      "linear_trend": pixel_linear_trend(serie)
  }


A saída de streaming foi truncada nas últimas 5000 linhas.
{'mean': np.float64(34.622047424316406), 'var': np.float64(0.0), 'max': np.float64(34.622047424316406), 'min': np.float64(34.622047424316406), 'linear_trend': 0.0}

PIXEL (np.int64(137), np.int64(77))
{'mean': np.float64(13.575945377349854), 'var': np.float64(0.0), 'max': np.float64(13.575945377349854), 'min': np.float64(13.575945377349854), 'linear_trend': 0.0}

PIXEL (np.int64(137), np.int64(78))
{'mean': np.float64(11.889473915100098), 'var': np.float64(0.0), 'max': np.float64(11.889473915100098), 'min': np.float64(11.889473915100098), 'linear_trend': 0.0}

PIXEL (np.int64(137), np.int64(79))
{'mean': np.float64(28.377368927001953), 'var': np.float64(0.0), 'max': np.float64(28.377368927001953), 'min': np.float64(28.377368927001953), 'linear_trend': 0.0}

PIXEL (np.int64(137), np.int64(80))
{'mean': np.float64(28.87661361694336), 'var': np.float64(0.0), 'max': np.float64(28.87661361694336), 'min': np.float64(28.87661361694336

In [ ]:
{}